# Build & publish the INDEPENDENT TEST set  (maximized from training sources)

Self-contained (only this notebook ships to Colab). Builds a **fixed, reproducible
benchmark** for the houseplant classifier by pulling **curated, folder-labelled**
images from the same Kaggle sources the training notebook uses, de-duping,
balancing per class, and **auto-creating/versioning** the private Kaggle dataset
`maxayyt5/houseplant-disease-testset`.

**Honesty (read this):** every source here is ALSO a *training* source, so this set
is **leaky** — absolute numbers are inflated. Its value is a **fixed set both models
run on**, so a *paired* comparison (model A vs B on identical images) is valid for
the shared/stable sources (leakage ~cancels in the delta). For a clean *generalization*
gate, pHash-dedup the rows against the training manifest, or swap in production photos.
Token is read from a **secret** (`KAGGLE_API_TOKEN`), never hardcoded.


In [ ]:
# === §1 Config ===========================================================
CLASS_NAMES = ['gray_mold_botrytis','healthy','leaf_scorch_underwatering','leaf_spot',
               'not_a_plant','nutrient_deficiency','pests','powdery_mildew','viral_disease']
TEST_SLUG       = 'maxayyt5/houseplant-disease-testset'
PER_CLASS_CAP   = 250      # maximize but keep classes balanced for macro-F1
IMG_MAX         = 768      # downscale long side
PHASH_THRESHOLD = 6        # Hamming <= -> near-dup (drop)
SEED            = 0
DO_UPLOAD       = True
OUT_DIR         = 'testset_build'
# Sources (all curated, folder-labelled, all TRAINING sources -> leaky).
USE_DOCTORP     = True     # alexanderuzhinskiy/the-doctorp-project-dataset (all classes)
USE_PEPPER      = True     # alexanderuzhinskiy/pepper-disease-classification-dataset-doctorp
USE_ORNAMENTAL  = True     # pappu54/indoor-plant-disease-dataset-23-classes (big)
USE_SUPPLEMENT  = True     # maxayyt5/houseplant-disease-eval-supplement (the only not_a_plant source)
USE_HOUSEPLANT  = False    # abdulahad0296/indoor-plant-disease-detection-dataset (2.2 GB; healthy/leaf_spot heavy)

In [ ]:
# === §2 Kaggle auth from a SECRET ========================================
import os
def _load_token():
    try:
        from google.colab import userdata
        t = userdata.get('KAGGLE_API_TOKEN')
        if t: return t.strip()
    except Exception: pass
    if os.environ.get('KAGGLE_API_TOKEN'): return os.environ['KAGGLE_API_TOKEN'].strip()
    p = os.path.expanduser('~/.kaggle/access_token')
    return open(p).read().strip() if os.path.exists(p) else None
_TOKEN=_load_token()
assert _TOKEN, "Add a Colab secret KAGGLE_API_TOKEN (or env / ~/.kaggle/access_token)."
import kagglehub
from kagglehub.config import set_kaggle_api_token
set_kaggle_api_token(_TOKEN); print('Kaggle token loaded (bearer).')

In [ ]:
# === §3 Helpers ==========================================================
import io, hashlib, json, numpy as np
from PIL import Image
def sha256_file(p, buf=1<<20):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        while (b:=f.read(buf)): h.update(b)
    return h.hexdigest()
_DCT=None
def _basis(n=32):
    k=np.arange(n); b=np.cos(np.pi*(2*k[:,None]+1)*k[None,:]/(2*n)); b[0]/=np.sqrt(2); return b*np.sqrt(2/n)
def phash(im):
    global _DCT
    if _DCT is None: _DCT=_basis(32)
    g=np.asarray(im.convert('L').resize((32,32),Image.BILINEAR),np.float64)
    low=(_DCT@g@_DCT.T)[:8,:8]; bits=(low>np.median(low.flatten()[1:])).flatten()
    h=0
    for x in bits: h=(h<<1)|int(x)
    return h
def hamming(a,b): return bin(a^b).count('1')
def load_img(p):
    try:
        im=Image.open(p).convert('RGB')
        if min(im.size)<96: return None
        im.thumbnail((IMG_MAX,IMG_MAX)); return im
    except Exception: return None

In [ ]:
# === §4 Curated source folder->class maps (copied from train_notebook) ====
import os
def _kw_none(): return None
DOCTORP_MAP={'grey mold':'gray_mold_botrytis','botrytis cinerea':'gray_mold_botrytis','scale':'pests',
 'mealybug':'pests','marginal leaf necrosis':'leaf_scorch_underwatering','burn':'leaf_scorch_underwatering',
 'leaves scorch':'leaf_scorch_underwatering','mosaic virus':'viral_disease','powdery mildew':'powdery_mildew',
 'nutrient deficiency':'nutrient_deficiency','healthy':'healthy','cherry leaf spot':'leaf_spot',
 'bacterial spot':'leaf_spot','leaf spot':'leaf_spot','anthracnose':'leaf_spot'}
PEPPER_MAP={'healthy':'healthy','nutrient deficiency':'nutrient_deficiency','bacterial spot':'leaf_spot',
 'mosaic virus':'viral_disease','powdery mildew':'powdery_mildew','burn':'leaf_scorch_underwatering'}
ORNAMENTAL_MAP={'rose_healthy':'healthy','rose_black_spot':'leaf_spot','money plant_money_plant_healthy':'healthy',
 'turmeric_healthy':'healthy','hibiscus_healthy':'healthy','hibiscus_scorch':'leaf_scorch_underwatering',
 'rose_mosaic_virus':'viral_disease','rose_powdery_mildew':'powdery_mildew','turmeric_leaf_spot':'leaf_spot',
 'chrysanthemum_bacterial_leaf_spot':'leaf_spot','chrysanthemum_healthy':'healthy',
 'chrysanthemum_septoria_leaf_spot':'leaf_spot','rose_yellow_mosaic_virus':'viral_disease',
 'money plant_money_plant_chlorosis':'nutrient_deficiency'}   # everything else -> DROP
HOUSEPLANT_MAP={'aloe_healthy':'healthy','aloe_leafspot':'leaf_spot','aloe_anthracnose':'leaf_spot',
 'cactus_healthy':'healthy','spider_plant_healthy':'healthy','spider_plant_leaf_tip_necrosis':'leaf_scorch_underwatering',
 'money_plant_healthy':'healthy','snake_plant_healthy':'healthy','spider_plant_fungal_leaf_spot':'leaf_spot',
 'snake_plant_anthracnose':'leaf_spot'}
# supplement folders ARE canonical class names already -> identity map.
SUPP_MAP={c:c for c in CLASS_NAMES}

def collect(slug, fmap):
    """Download a Kaggle dataset and return {class: [image paths]} via folder->class map."""
    root=kagglehub.dataset_download(slug); out={}
    for dp,_,files in os.walk(root):
        cls=fmap.get(os.path.basename(dp).strip().lower())
        if not cls: continue
        for f in files:
            if f.lower().endswith(('.jpg','.jpeg','.png')):
                out.setdefault(cls,[]).append(os.path.join(dp,f))
    return out

SOURCES=[]
if USE_DOCTORP:    SOURCES.append(('doctorp','alexanderuzhinskiy/the-doctorp-project-dataset',DOCTORP_MAP))
if USE_PEPPER:     SOURCES.append(('doctorp_pepper','alexanderuzhinskiy/pepper-disease-classification-dataset-doctorp',PEPPER_MAP))
if USE_ORNAMENTAL: SOURCES.append(('kaggle_ornamental','pappu54/indoor-plant-disease-dataset-23-classes',ORNAMENTAL_MAP))
if USE_SUPPLEMENT: SOURCES.append(('supplement','maxayyt5/houseplant-disease-eval-supplement',SUPP_MAP))
if USE_HOUSEPLANT: SOURCES.append(('kaggle_houseplant','abdulahad0296/indoor-plant-disease-detection-dataset',HOUSEPLANT_MAP))

In [ ]:
# === §5 Build: gather -> sha-order -> global dedup -> cap -> downscale =====
import shutil, csv
shutil.rmtree(OUT_DIR, ignore_errors=True)
# gather candidates per class, tagged with source
cands={c:[] for c in CLASS_NAMES}
for sid, slug, fmap in SOURCES:
    print('downloading', slug, '...')
    got=collect(slug, fmap)
    for c, paths in got.items():
        cands[c].extend((p, sid) for p in paths)
    print('  ', {c:len(v) for c,v in got.items()})
rows, counts, seen_ph = [], {c:0 for c in CLASS_NAMES}, []
SRC_URL={'doctorp':'kaggle.com/datasets/alexanderuzhinskiy/the-doctorp-project-dataset',
 'doctorp_pepper':'kaggle.com/datasets/alexanderuzhinskiy/pepper-disease-classification-dataset-doctorp',
 'kaggle_ornamental':'kaggle.com/datasets/pappu54/indoor-plant-disease-dataset-23-classes',
 'supplement':'kaggle.com/datasets/maxayyt5/houseplant-disease-eval-supplement',
 'kaggle_houseplant':'kaggle.com/datasets/abdulahad0296/indoor-plant-disease-detection-dataset'}
for c in CLASS_NAMES:
    import hashlib as _hl
    items=cands[c]
    items.sort(key=lambda t: _hl.md5(t[0].encode()).hexdigest())   # deterministic, no file read
    for p, sid in items:
        if counts[c]>=PER_CLASS_CAP: break
        im=load_img(p)
        if im is None: continue
        ph=phash(im)
        if any(hamming(ph,h)<=PHASH_THRESHOLD for h in seen_ph): continue
        seen_ph.append(ph)
        d=f'{OUT_DIR}/{c}'; os.makedirs(d,exist_ok=True)
        fn=f'{c}_{counts[c]:03d}.jpg'; im.save(f'{d}/{fn}',quality=88); counts[c]+=1
        rows.append({'file':f'{c}/{fn}','class':c,'source':sid,'source_url':SRC_URL.get(sid,''),
                     'note':'TRAINING source (leaky) - dedup vs train before absolute claims'})
with open(f'{OUT_DIR}/manifest.csv','w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=['file','class','source','source_url','note']); w.writeheader(); w.writerows(rows)
total=sum(counts.values())
import collections
bysrc=collections.Counter(r['source'] for r in rows)
open(f'{OUT_DIR}/README.md','w').write(
 f"# Houseplant disease - fixed benchmark ({total} imgs, {sum(1 for c in counts if counts[c])} classes)\n\n"
 f"Per-class: {json.dumps(counts)}\n\nBy source: {json.dumps(dict(bysrc))}\n\n"
 f"ALL sources are TRAINING sources -> this set is LEAKY. Use it for PAIRED model "
 f"comparison on identical images (delta valid for shared/stable sources); for a clean "
 f"generalization gate, pHash-dedup vs the training manifest or use production photos.\n")
print('BUILT', total, '| per-class', json.dumps(counts), '| by source', dict(bysrc))

In [ ]:
# === §6 Auto-create / update the private Kaggle dataset ===================
if DO_UPLOAD:
    kagglehub.dataset_upload(TEST_SLUG, OUT_DIR,
        version_notes=f'max-from-sources: {sum(counts.values())} imgs / '
                      f'{sum(1 for c in counts if counts[c])} classes (curated training sources, leaky)')
    print('uploaded ->', f'https://kaggle.com/datasets/{TEST_SLUG}')
else:
    print('DO_UPLOAD=False -> built locally only.')

In [ ]:
# === §7 (optional) Validate an exported bundle against this benchmark =====
BUNDLE_DIR='models'; RUN_VALIDATION=False
if RUN_VALIDATION:
    import onnxruntime as ort
    from sklearn.metrics import f1_score
    meta=json.load(open(f'{BUNDLE_DIR}/classes.json')); cn=meta['class_names']
    size=(meta.get('training') or {}).get('img_size',224); ratio=(meta.get('training') or {}).get('eval_resize_ratio',1.05)
    mean=np.array([.485,.456,.406],np.float32); std=np.array([.229,.224,.225],np.float32)
    onx=next((f'{BUNDLE_DIR}/{n}' for n in ('classifier_int8.onnx','classifier.onnx') if os.path.exists(f'{BUNDLE_DIR}/{n}')),None)
    s=ort.InferenceSession(onx,providers=['CPUExecutionProvider']); inp=s.get_inputs()[0].name
    def prep(im):
        z=max(size,int(size*ratio)); w,h=im.size; sc=z/min(w,h)
        im=im.resize((max(size,round(w*sc)),max(size,round(h*sc))),Image.BILINEAR)
        l=(im.size[0]-size)//2; t=(im.size[1]-size)//2; im=im.crop((l,t,l+size,t+size))
        return (((np.asarray(im,np.float32)/255-mean)/std).transpose(2,0,1)[None])
    yt,yp=[],[]
    for ci,c in enumerate(cn):
        d=f'{OUT_DIR}/{c}'
        if not os.path.isdir(d): continue
        for fn in os.listdir(d):
            yp.append(int(s.run(None,{inp:prep(Image.open(f'{d}/{fn}').convert('RGB'))})[0].argmax())); yt.append(ci)
    yt,yp=np.array(yt),np.array(yp); di=[i for i,n in enumerate(cn) if n not in ('healthy','not_a_plant')]
    f1=f1_score(yt,yp,labels=di,average='macro',zero_division=0); r=np.random.default_rng(SEED)
    bs=[f1_score(yt[x],yp[x],labels=di,average='macro',zero_division=0) for x in (r.integers(0,len(yt),len(yt)) for _ in range(2000))]
    print(f'BENCHMARK disease_macro_f1={f1:.3f} 95% CI [{np.percentile(bs,2.5):.3f},{np.percentile(bs,97.5):.3f}] n={len(yt)}')